# Cuff-less Blood Pressure Estimation: Random Forest on PPG Features

**Program:** Break Through Tech AI Studio, Fall 2026 | **Challenge:** CufflessAI (Microsoft)

This notebook builds an **individual Random Forest regression model** that estimates **systolic (SBP)** and **diastolic (DBP)** blood pressure from **PPG (photoplethysmogram)** signal windows.

**Pipeline**
1. Configure paths and settings
2. Inspect the `Part_1.mat` (HDF5 / MATLAB v7.3) structure
3. Load PPG and ABP signals, keeping them aligned
4. Cut the signals into aligned 5-second windows (625 samples at 125 Hz)
5. Compute SBP/DBP labels from the ABP window and discard noisy windows
6. Extract numerical features from each PPG window
7. Split train/test **by recording** (to limit data leakage)
8. Train a `RandomForestRegressor` and compare it to a naive baseline
9. Evaluate with MAE, RMSE, and R²
10. Plot actual vs. predicted, residuals, and feature importance
11. Save the model, metrics, predictions, and figures

> **Educational prototype only.** This model is not a medical device and must never be used for health decisions.
> No results are shown in this notebook until **you** run it on your own copy of the data.

## 0. Setup

Run the install cell once (Colab usually needs `h5py` only). Locally you can instead run `pip install -r requirements.txt`.

In [ ]:
# Install dependencies (safe to re-run). Skip this cell if you already installed requirements.txt.
%pip install -q numpy pandas scipy scikit-learn h5py matplotlib joblib tqdm

In [ ]:
import os, sys, json, time, platform
import numpy as np
import pandas as pd
import h5py
import joblib
import matplotlib.pyplot as plt
import sklearn
from scipy.signal import find_peaks, butter, filtfilt
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

try:
    from tqdm.auto import tqdm
except ImportError:                      # tqdm is optional
    tqdm = lambda x, **k: x

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| h5py", h5py.__version__)

## 1. Configuration

Everything you may want to change is in this one cell.

* **`DATA_PATH`**: location of `Part_1.mat`.
  * Local: e.g. `"data/Part_1.mat"`
  * Colab: upload the file to the session (left sidebar → Files), or mount Google Drive in the next cell and use e.g. `"/content/drive/MyDrive/Part_1.mat"`.
* **`MAX_RECORDINGS`**: how many recordings to use. Part_1 has about 3000 records, so we start with a random subset. Set to `None` to use all of them (slower).
* **`MAX_WINDOWS_PER_RECORDING`**: caps windows taken from each recording (evenly spaced) so a few very long recordings do not dominate. Set to `None` for no cap.

In [ ]:
# ----------------------------- CONFIG -----------------------------
DATA_PATH = "Part_1 (1).mat"        # <-- change to your file location
MAT_KEY = None                  # top-level variable name in the .mat file; None = auto-detect (expected: "Part_1")

FS = 125                        # sampling frequency (Hz), per dataset description
WINDOW_SECONDS = 5
WINDOW_SAMPLES = FS * WINDOW_SECONDS   # 625

# Channel order inside each record. VERIFY with the inspection cell below.
# UCI description: row 1 = PPG, row 2 = ABP (mmHg), row 3 = ECG.
CH_PPG, CH_ABP, CH_ECG = 0, 1, 2

MAX_RECORDINGS = 300            # start small; None = all recordings
MAX_WINDOWS_PER_RECORDING = 40  # None = no cap
RANDOM_SEED = 42
TEST_SIZE = 0.2                 # fraction of RECORDINGS held out for testing

# Label-quality limits (physiologically plausible ranges, mmHg)
SBP_RANGE = (60, 250)
DBP_RANGE = (30, 150)
MIN_PULSE_PRESSURE = 10         # SBP - DBP must be at least this

N_ESTIMATORS = 200
USE_CACHE = True                # reuse extracted features if the same settings were already processed
OUTPUT_DIR = "outputs"
# ------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
np.random.seed(RANDOM_SEED)
print("Window length:", WINDOW_SAMPLES, "samples |", "Data path:", os.path.abspath(DATA_PATH))

In [ ]:
# OPTIONAL (Google Colab only): mount Google Drive if your data lives there.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted. Set DATA_PATH to something like /content/drive/MyDrive/.../Part_1.mat")
else:
    print("Not running in Colab; skipping Drive mount.")

## 2. Inspect the HDF5 structure (verify assumptions)

MATLAB v7.3 files are HDF5 files. The dataset is a **cell array**: the top-level variable (expected name `Part_1`) holds a list of **HDF5 object references**. Each reference points to a separate matrix for one record.

Because MATLAB stores arrays column-major, `h5py` often shows each record **transposed** compared with MATLAB, i.e. shape `(n_samples, 3)` instead of `(3, n_samples)`. The loader below handles both orientations.

**Assumptions to verify from the output of the next cell:**
1. The top-level key is `Part_1` (or the one you set in `MAT_KEY`).
2. Each record has 3 channels in the order PPG, ABP, ECG.
3. The ABP channel looks like mmHg (typically values between ~50 and ~200).
4. The PPG channel is a smooth pulse waveform.

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find {DATA_PATH!r}. Update DATA_PATH in the config cell.")

with h5py.File(DATA_PATH, "r") as f:
    print("Top-level keys:", list(f.keys()))
    candidates = [k for k in f.keys() if not k.startswith("#")]   # '#refs#' is HDF5 bookkeeping
    key = MAT_KEY or candidates[0]
    cell = f[key]
    print(f"Using key {key!r}: shape={cell.shape}, dtype={cell.dtype}")
    refs = cell[()]
    print("Number of record references:", refs.size)

    for i in range(min(3, refs.size)):
        rec = np.asarray(f[refs.ravel()[i]])
        print(f"\nRecord {i}: raw h5py shape = {rec.shape}")
        sig = rec if rec.shape[1] == 3 else rec.T          # (n_samples, 3)
        for name, ch in (("PPG", CH_PPG), ("ABP", CH_ABP), ("ECG", CH_ECG)):
            x = sig[:, ch]
            print(f"  {name} (channel {ch}): min={x.min():.2f}  median={np.median(x):.2f}  max={x.max():.2f}")
        print(f"  Duration: {sig.shape[0]/FS:.1f} s")

## 3. Load recordings

`load_recordings` follows each HDF5 reference, turns the record into a `(n_samples, 3)` array, and keeps **only PPG and ABP** (same index = same time point, so they stay aligned).

We pick a reproducible random subset of recordings. Each recording keeps its **original index** (`record_id`), which we use later to split by recording.

In [ ]:
def orient_record(arr):
    """Return the record as (n_samples, 3), whichever way h5py/MATLAB stored it."""
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2-D matrix per record, got shape {arr.shape}")
    if arr.shape[1] == 3:
        return arr
    if arr.shape[0] == 3:
        return arr.T
    raise ValueError(f"Cannot find a 3-channel axis in shape {arr.shape}")


def load_recordings(path, key=None, max_recordings=None, seed=42):
    """Load PPG and ABP (aligned) for a random subset of recordings.

    Returns a list of dicts: {"record_id": int, "ppg": 1-D array, "abp": 1-D array}
    """
    recordings = []
    with h5py.File(path, "r") as f:
        candidates = [k for k in f.keys() if not k.startswith("#")]
        k = key or candidates[0]
        obj = f[k]
        if getattr(obj, "dtype", None) is None or obj.dtype.kind != "O":
            raise TypeError(f"{k!r} is not a cell array of references (dtype={getattr(obj, 'dtype', None)}). "
                            "Check MAT_KEY and the inspection cell above.")
        refs = obj[()].ravel()
        n_total = len(refs)
        idx = np.arange(n_total)
        if max_recordings is not None and max_recordings < n_total:
            idx = np.sort(np.random.default_rng(seed).choice(n_total, size=max_recordings, replace=False))
        for i in tqdm(idx, desc="Loading recordings"):
            sig = orient_record(f[refs[i]])
            ppg, abp = sig[:, CH_PPG], sig[:, CH_ABP]          # same row index = same time stamp
            n = min(len(ppg), len(abp))
            recordings.append({"record_id": int(i), "ppg": ppg[:n], "abp": abp[:n]})
    print(f"Loaded {len(recordings)} of {n_total} recordings from {path!r} using key {k!r}")
    return recordings

## 4. Windowing, labels, and feature extraction

### Windows (aligned)
We cut each recording into **non-overlapping 625-sample windows** (5 s × 125 Hz). The same start/end indices are used for PPG and ABP, so they remain aligned.

### Labels from ABP
For each ABP window:
* **SBP** = median of the pressure peaks (one per heartbeat)
* **DBP** = median of the pressure valleys

A window is **discarded** if the ABP has NaN values, too few beats, or values outside plausible ranges (see config). This removes noisy or flat segments.

### Features from PPG
Per window we compute:
* **Statistical features:** mean, std, min, max, range, median, percentiles, skewness, kurtosis, RMS, mean absolute change (variability)
* **Derivative features:** statistics of the first derivative (signal slope)
* **Beat features** (from a 0.5–8 Hz band-pass filtered copy): number of beats, heart rate, inter-beat interval mean/std/RMSSD, pulse amplitude, rise time
* **Spectral features:** dominant frequency and fraction of power in the cardiac band

> The window is also dropped if the PPG is flat/saturated or if fewer than 3 PPG beats can be found.

In [ ]:
# ---------- ABP -> labels ----------
def bp_labels(abp_win):
    """Return (SBP, DBP) from an ABP window, or None if the window is unusable."""
    if not np.all(np.isfinite(abp_win)):
        return None
    peaks, _ = find_peaks(abp_win, distance=int(0.3 * FS), prominence=10)     # systolic peaks
    valleys, _ = find_peaks(-abp_win, distance=int(0.3 * FS), prominence=10)  # diastolic valleys
    if len(peaks) < 3 or len(valleys) < 3:
        return None
    sbp = float(np.median(abp_win[peaks]))
    dbp = float(np.median(abp_win[valleys]))
    if not (SBP_RANGE[0] <= sbp <= SBP_RANGE[1] and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]):
        return None
    if sbp - dbp < MIN_PULSE_PRESSURE:
        return None
    return sbp, dbp


# ---------- PPG quality + features ----------
_b, _a = butter(3, [0.5, 8.0], btype="bandpass", fs=FS)

def ppg_is_usable(ppg_win):
    if not np.all(np.isfinite(ppg_win)):
        return False
    if np.std(ppg_win) < 1e-6:
        return False
    if np.mean(np.diff(ppg_win) == 0) > 0.2:      # >20% repeated samples = flat-lined / clipped
        return False
    return True


def ppg_features(ppg_win):
    """Compute a dict of numerical features for one PPG window, or None if beats cannot be found."""
    x = ppg_win.astype(np.float64)
    d = np.diff(x)
    feats = {
        "mean": x.mean(), "std": x.std(), "min": x.min(), "max": x.max(),
        "range": np.ptp(x), "median": np.median(x),
        "p10": np.percentile(x, 10), "p25": np.percentile(x, 25),
        "p75": np.percentile(x, 75), "p90": np.percentile(x, 90),
        "skew": skew(x), "kurtosis": kurtosis(x),
        "rms": np.sqrt(np.mean(x ** 2)),
        "mean_abs_diff": np.mean(np.abs(d)),             # variability
        "d1_mean": d.mean(), "d1_std": d.std(), "d1_max": d.max(), "d1_min": d.min(),
    }

    # Beat-level features from a band-passed copy
    filt = filtfilt(_b, _a, x)
    fstd = filt.std()
    if fstd < 1e-9:
        return None
    peaks, _ = find_peaks(filt, distance=int(0.3 * FS), prominence=0.3 * fstd)
    troughs, _ = find_peaks(-filt, distance=int(0.3 * FS), prominence=0.3 * fstd)
    if len(peaks) < 3:
        return None

    ibi = np.diff(peaks) / FS                              # inter-beat intervals (s)
    feats.update({
        "n_beats": len(peaks),
        "hr_bpm": 60.0 / ibi.mean(),
        "ibi_mean": ibi.mean(),
        "ibi_std": ibi.std(),
        "ibi_rmssd": np.sqrt(np.mean(np.diff(ibi) ** 2)) if len(ibi) > 1 else 0.0,
    })

    amps, rises = [], []
    for p in peaks:
        prev_tr = troughs[troughs < p]
        if len(prev_tr):
            t = prev_tr[-1]
            amps.append(filt[p] - filt[t])
            rises.append((p - t) / FS)
    feats["pulse_amp_mean"] = float(np.mean(amps)) if amps else np.nan
    feats["pulse_amp_std"] = float(np.std(amps)) if amps else np.nan
    feats["rise_time_mean"] = float(np.mean(rises)) if rises else np.nan
    feats["rise_to_ibi"] = float(np.mean(rises) / ibi.mean()) if rises else np.nan
    feats["peak_height_mean"] = float(filt[peaks].mean())

    # Spectral features
    spec = np.abs(np.fft.rfft(filt - filt.mean())) ** 2
    freqs = np.fft.rfftfreq(len(filt), d=1.0 / FS)
    band = (freqs >= 0.5) & (freqs <= 4.0)
    feats["dom_freq"] = float(freqs[band][np.argmax(spec[band])])
    feats["cardiac_power_ratio"] = float(spec[band].sum() / (spec.sum() + 1e-12))

    return {k: float(v) for k, v in feats.items()}


# ---------- Build the window dataset ----------
def build_dataset(recordings, window=WINDOW_SAMPLES, max_windows=MAX_WINDOWS_PER_RECORDING):
    rows, stats = [], {"windows_seen": 0, "bad_abp": 0, "bad_ppg": 0, "no_beats": 0, "kept": 0}
    for rec in tqdm(recordings, desc="Extracting features"):
        ppg, abp = rec["ppg"], rec["abp"]
        n_win = len(ppg) // window
        win_ids = np.arange(n_win)
        if max_windows is not None and n_win > max_windows:
            win_ids = np.unique(np.linspace(0, n_win - 1, max_windows).astype(int))   # evenly spaced
        for w in win_ids:
            s, e = w * window, (w + 1) * window                 # SAME indices for PPG and ABP => aligned
            stats["windows_seen"] += 1
            labels = bp_labels(abp[s:e])
            if labels is None:
                stats["bad_abp"] += 1; continue
            if not ppg_is_usable(ppg[s:e]):
                stats["bad_ppg"] += 1; continue
            feats = ppg_features(ppg[s:e])
            if feats is None:
                stats["no_beats"] += 1; continue
            feats.update({"record_id": rec["record_id"], "window_idx": int(w), "SBP": labels[0], "DBP": labels[1]})
            rows.append(feats)
            stats["kept"] += 1
    df = pd.DataFrame(rows)
    print("Window summary:", stats)
    return df

## 5. Build (or load cached) feature table

This is the slowest step. The result is cached to a CSV named after your settings, so re-running the notebook is fast.

In [ ]:
cache_name = f"features_n{MAX_RECORDINGS}_w{WINDOW_SAMPLES}_c{MAX_WINDOWS_PER_RECORDING}_s{RANDOM_SEED}.csv"
cache_path = os.path.join(OUTPUT_DIR, cache_name)

if USE_CACHE and os.path.exists(cache_path):
    df = pd.read_csv(cache_path)
    print(f"Loaded cached features from {cache_path}")
else:
    t0 = time.time()
    recordings = load_recordings(DATA_PATH, key=MAT_KEY, max_recordings=MAX_RECORDINGS, seed=RANDOM_SEED)
    df = build_dataset(recordings)
    del recordings
    # Replace any NaN beat features by the column median so no window is lost needlessly
    feat_cols_tmp = [c for c in df.columns if c not in ("record_id", "window_idx", "SBP", "DBP")]
    df[feat_cols_tmp] = df[feat_cols_tmp].fillna(df[feat_cols_tmp].median())
    df.to_csv(cache_path, index=False)
    print(f"Saved features to {cache_path} ({time.time() - t0:.0f}s)")

FEATURE_COLS = [c for c in df.columns if c not in ("record_id", "window_idx", "SBP", "DBP")]
print(f"\n{len(df)} windows from {df['record_id'].nunique()} recordings | {len(FEATURE_COLS)} features")
if len(df) < 200:
    print("WARNING: very few windows survived filtering. Check channel order / units in the inspection cell.")
df.head()

In [ ]:
# Quick look at the label distribution
print(df[["SBP", "DBP"]].describe().round(1))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, col in zip(axes, ["SBP", "DBP"]):
    ax.hist(df[col], bins=50, color="steelblue", edgecolor="white")
    ax.set_title(f"{col} distribution (mmHg)"); ax.set_xlabel("mmHg"); ax.set_ylabel("windows")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "label_distribution.png"), dpi=150)
plt.show()

## 6. Train/test split by recording (avoiding leakage)

Windows from the same recording look very similar. If some went to training and others to testing, the model could "memorize" the recording and the test score would look better than reality.

We use `GroupShuffleSplit` with **`record_id` as the group**, so each recording appears in either train **or** test, never both.

> ### ⚠️ Important limitation
> `Part_1.mat` contains **no patient IDs**. A recording-level split does **not guarantee patient independence**: the same patient may have contributed several recordings, some of which could land in train and others in test. The reported scores may therefore be somewhat optimistic compared with a true unseen-patient evaluation. Say this explicitly when presenting results.

In [ ]:
X = df[FEATURE_COLS].values
y = df[["SBP", "DBP"]].values
groups = df["record_id"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_recs, test_recs = set(groups[train_idx]), set(groups[test_idx])
assert train_recs.isdisjoint(test_recs), "Leakage: a recording appears in both train and test!"
print(f"Train: {len(train_idx)} windows from {len(train_recs)} recordings")
print(f"Test : {len(test_idx)} windows from {len(test_recs)} recordings")
print("No recording appears in both sets. (Patient-level independence is NOT guaranteed; see note above.)")

## 7. Train the Random Forest and a naive baseline

* **Baseline:** always predicts the *training-set mean* SBP and DBP. Any useful model must beat this.
* **Random Forest:** one `RandomForestRegressor` with two outputs (SBP, DBP). Random Forests need no feature scaling.

In [ ]:
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

rf = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=5,        # smoother trees, less overfitting
    max_features=0.5,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
t0 = time.time()
rf.fit(X_train, y_train)
print(f"Random Forest trained in {time.time() - t0:.1f}s")

## 8. Evaluation (MAE, RMSE, R²)

* **MAE**: average absolute error in mmHg (lower is better)
* **RMSE**: like MAE but penalizes large errors more (lower is better)
* **R²**: share of variance explained (1 = perfect, 0 = no better than predicting the mean, negative = worse)

In [ ]:
def evaluate(y_true, y_pred, model_name):
    rows = []
    for j, target in enumerate(["SBP", "DBP"]):
        rows.append({
            "Model": model_name, "Target": target,
            "MAE (mmHg)": mean_absolute_error(y_true[:, j], y_pred[:, j]),
            "RMSE (mmHg)": np.sqrt(mean_squared_error(y_true[:, j], y_pred[:, j])),
            "R2": r2_score(y_true[:, j], y_pred[:, j]),
        })
    return rows

pred_base = baseline.predict(X_test)
pred_rf = rf.predict(X_test)

results = pd.DataFrame(evaluate(y_test, pred_base, "Baseline (train mean)") + evaluate(y_test, pred_rf, "Random Forest"))
results = results.round(3)
results

## 9. Plots: actual vs. predicted
Points close to the dashed diagonal indicate accurate predictions. If the cloud is flat or horizontal, the model is mostly predicting the average.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for j, target in enumerate(["SBP", "DBP"]):
    ax = axes[0, j]
    ax.scatter(y_test[:, j], pred_rf[:, j], s=6, alpha=0.3, color="steelblue")
    lo, hi = min(y_test[:, j].min(), pred_rf[:, j].min()), max(y_test[:, j].max(), pred_rf[:, j].max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="perfect prediction")
    m = results[(results.Model == "Random Forest") & (results.Target == target)].iloc[0]
    ax.set_title(f"{target}: actual vs predicted\nMAE={m['MAE (mmHg)']:.2f}  RMSE={m['RMSE (mmHg)']:.2f}  R²={m['R2']:.3f}")
    ax.set_xlabel("Actual (mmHg)"); ax.set_ylabel("Predicted (mmHg)"); ax.legend()

    ax = axes[1, j]
    resid = pred_rf[:, j] - y_test[:, j]
    ax.hist(resid, bins=50, color="indianred", edgecolor="white")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(f"{target}: residuals (predicted - actual)")
    ax.set_xlabel("Error (mmHg)"); ax.set_ylabel("windows")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "actual_vs_predicted.png"), dpi=150)
plt.show()

In [ ]:
# Model vs baseline (MAE)
pivot = results.pivot(index="Target", columns="Model", values="MAE (mmHg)")
ax = pivot.plot(kind="bar", figsize=(6, 4), rot=0, color=["gray", "steelblue"])
ax.set_ylabel("MAE (mmHg)"); ax.set_title("MAE: Random Forest vs naive baseline (test set)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "mae_vs_baseline.png"), dpi=150)
plt.show()

## 10. Feature importance

Two views:
* **Impurity-based importance** (built into Random Forests; quick, but biased toward features with many distinct values).
* **Permutation importance** on the **test set** (how much the score drops when a feature is shuffled; more reliable).

In [ ]:
imp = pd.DataFrame({"feature": FEATURE_COLS, "impurity_importance": rf.feature_importances_})

perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1)
imp["permutation_importance"] = perm.importances_mean
imp = imp.sort_values("permutation_importance", ascending=False).reset_index(drop=True)

top = imp.head(15).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(top["feature"], top["impurity_importance"], color="steelblue"); axes[0].set_title("Impurity importance (top 15 by permutation rank)")
axes[1].barh(top["feature"], top["permutation_importance"], color="darkorange"); axes[1].set_title("Permutation importance on test set")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "feature_importance.png"), dpi=150)
plt.show()
imp.head(15).round(4)

## 11. Save the model and results

Saved into `OUTPUT_DIR` (default `outputs/`):
* `rf_bp_model.joblib`: trained model + feature names + settings
* `metrics.csv` / `metrics.json`: evaluation table
* `predictions_test.csv`: actual and predicted values for the test windows
* `feature_importance.csv`
* `figures/*.png`

In [ ]:
settings = {
    "DATA_PATH": DATA_PATH, "FS": FS, "WINDOW_SAMPLES": WINDOW_SAMPLES,
    "MAX_RECORDINGS": MAX_RECORDINGS, "MAX_WINDOWS_PER_RECORDING": MAX_WINDOWS_PER_RECORDING,
    "RANDOM_SEED": RANDOM_SEED, "TEST_SIZE": TEST_SIZE, "N_ESTIMATORS": N_ESTIMATORS,
    "n_windows": int(len(df)), "n_train_recordings": len(train_recs), "n_test_recordings": len(test_recs),
    "sklearn_version": sklearn.__version__,
}

joblib.dump({"model": rf, "feature_names": FEATURE_COLS, "targets": ["SBP", "DBP"], "settings": settings},
            os.path.join(OUTPUT_DIR, "rf_bp_model.joblib"))

results.to_csv(os.path.join(OUTPUT_DIR, "metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
    json.dump({"settings": settings, "metrics": results.to_dict(orient="records")}, fh, indent=2)

pred_df = pd.DataFrame({
    "record_id": groups[test_idx], "window_idx": df["window_idx"].values[test_idx],
    "SBP_actual": y_test[:, 0], "SBP_pred": pred_rf[:, 0],
    "DBP_actual": y_test[:, 1], "DBP_pred": pred_rf[:, 1],
})
pred_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_test.csv"), index=False)
imp.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)

print("Saved:", sorted(os.listdir(OUTPUT_DIR)))

In [ ]:
# Example: reload the saved model and predict on a few test windows
bundle = joblib.load(os.path.join(OUTPUT_DIR, "rf_bp_model.joblib"))
sample = df.iloc[test_idx[:5]][bundle["feature_names"]].values
print(pd.DataFrame(bundle["model"].predict(sample), columns=bundle["targets"]).round(1))

## 12. Notes, limitations, and ideas to try next

* **Not clinical.** Success here means a complete, responsible, reproducible pipeline, not clinical accuracy.
* **No patient IDs in Part_1.mat.** Recording-level splitting reduces but does not remove leakage; scores may be optimistic.
* **Subset.** By default only a random subset of recordings is used. Increase `MAX_RECORDINGS` (or set it to `None`) for the final run, and report which setting you used.
* **Label noise.** SBP/DBP come from ABP peak/valley detection; thresholds in the config affect which windows survive.
* **Window length.** Change `WINDOW_SECONDS` to 3 or 10 to compare with your earlier experiments (the code uses `WINDOW_SAMPLES = FS * WINDOW_SECONDS`).
* **Ideas:** hyperparameter search with `GroupKFold`, NeuroKit2/pyPPG morphological features (e.g. dicrotic notch, second-derivative waves), a gradient-boosting comparison, adding a blood-pressure category classifier, error bands from per-tree predictions.